In [6]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "test"))


In [9]:
from src.demos.ekg.pydantic_rag import PydanticRag

KV_STORE = "file"


vector_store = PydanticRag.get_vector_store()

rag = PydanticRag(
    model_definition=test_schema,
    vector_store=vector_store,
    llm_id=None,
    kv_store_id=KV_STORE,
)


TypeError: missing a required argument: 'postgres_url'

In [ ]:
# A tiny markdown document

doc_text = """
# Jane Doe
Jane is 29 years old and can be reached at jane.doe@example.com (personal adress) and J.doe@apple.com (professional email).
She lives rue de la Concorde, 31000 Toulouse France.

"""
doc_id = "test"
key = "example Jane Joe #1"
# 1. Analyse → structured Pydantic object

person = rag.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", person)

assert person


2025-08-12 16:32:49.833 | DEBUG    | src.utils.pydantic.kv_store:load_object_from_kvstore:89 - read 'Person/test.json' from KV store


Structured result:
Person(
    name='Jane Doe',
    age=29,
    email=[
        Email(url='jane.doe@example.com', email_type='personal'),
        Email(url='J.doe@apple.com', email_type='professional')
    ],
    address=Address(street='rue de la Concorde', city='Toulouse', zip_code='31000', country='France'),
    doc_id='test'
)

In [ ]:
chunks = rag.chunck(person)


In [ ]:
debug(chunks)

In [ ]:
# 2. Index the document
# rag.store_chunks(chunks)
print("Document stored.")


In [ ]:
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

In [ ]:
# 3. Query the vector store
hits = rag.query_vectorstore(
    "e-mail address", k=2, filter={"langchain_metadata ->> description": {"$eq": "Age in years"}}
)
print("Vector hits:", hits)